In [33]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

joblib — saves the fitted pipeline to disk. The API will load this to make predictions on new data
ColumnTransformer — applies different transformations to different columns simultaneously
SimpleImputer — fills missing values automatically.

In [34]:
df = pd.read_csv("../data/processed/telco_churn_cleaned.csv")
print(df.shape)
print(df.columns.tolist())
print(f"churn distribution:\n{df['Churn'].value_counts()}")

(7043, 20)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [35]:
# print(df['OnlineSecurity'].value_counts())
df['OnlineSecurity'] = df['OnlineSecurity'].replace('No internet service', 'No')
# print(df['OnlineSecurity'].value_counts())
# print(df['OnlineBackup'].value_counts())
df['OnlineBackup'] = df['OnlineBackup'].replace('No internet service', 'No')
# print(df['OnlineBackup'].value_counts())
# print(df['MultipleLines'].value_counts())
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')
df['DeviceProtection'] = df['OnlineBackup'].replace('No internet service', 'No')
df['TechSupport'] = df['OnlineBackup'].replace('No internet service', 'No')
df['StreamingTV'] = df['OnlineBackup'].replace('No internet service', 'No')
df['StreamingMovies'] = df['OnlineBackup'].replace('No internet service', 'No')


In [36]:
X = df.drop(columns="Churn")
y = df['Churn']

print(f"Features shape: {X.shape}")
print(f"Target Distribution\n{y.value_counts(normalize=True).round(3)}")

Features shape: (7043, 19)
Target Distribution
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64


In [37]:
X.dtypes

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
dtype: object

In [38]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = [col for col in X.columns if col not in numeric_features]

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features: {len(categorical_features)}: {categorical_features}")

Numeric features (4): ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical features: 15: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [39]:
# preprocessor building

# Numeric pipeline: fill missing values -> scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="median")), # median = robust to outliers
    ('scaler', StandardScaler())
])

# Categorical pipeline: fill missing values -> encode(OneHot)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy="most_frequent")), # mode for categories
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False))
])

# ColumnTransformer: glues both pipelines together
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'SeniorCitizen']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(drop='if_binary',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Partner', 'Dependents',
                    

**Why sparse_output=False: By default OHE returns a sparse matrix (memory efficient for huge datasets). We use False here for readability. For 7K rows it makes no practical difference.**

**Why handle_unknown='ignore': If a client's data has a new PaymentMethod type you've never seen, the API won't crash — it just outputs zeros for that column.**

In [40]:
# Train/Test split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"testing set: {X_test.shape[0]} rows")
print(f"churn rate in training set: {y_train.mean():.3f}")
print(f"churn rate in test set: {y_train.mean():.3f}")

Training set: 5634 rows
testing set: 1409 rows
churn rate in training set: 0.265
churn rate in test set: 0.265


stratify=y: Without it, random chance could put 90% of churners in train and 10% in test (or vice versa). Stratify guarantees both sets have the same 26.5% churn rate. Essential with imbalanced datasets.

In [41]:
# Fit and Transform

# here we will use our custom transformer to fit and transform our training data.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train before: {X_train.shape}  →  after: {X_train_processed.shape}")
print(f"X_test before:  {X_test.shape}  →  after: {X_test_processed.shape}")

X_train before: (5634, 19)  →  after: (5634, 26)
X_test before:  (1409, 19)  →  after: (1409, 26)


In [42]:
# Get feature names after OHE expansion

# again using custom transformer to get expanded feature names
ohe_feature_names = preprocessor.named_transformers_['cat']['onehot']\
.get_feature_names_out(categorical_features).tolist()

all_feature_names = numeric_features + ohe_feature_names

print(f"Total features after encoding: {len(all_feature_names)}")
print(f"\nFirst 15 feature names:")
for name in all_feature_names[:]:
    print(f"  {name}")

Total features after encoding: 26

First 15 feature names:
  tenure
  MonthlyCharges
  TotalCharges
  SeniorCitizen
  gender_Male
  Partner_Yes
  Dependents_Yes
  PhoneService_Yes
  MultipleLines_Yes
  InternetService_DSL
  InternetService_Fiber optic
  InternetService_No
  OnlineSecurity_Yes
  OnlineBackup_Yes
  DeviceProtection_Yes
  TechSupport_Yes
  StreamingTV_Yes
  StreamingMovies_Yes
  Contract_Month-to-month
  Contract_One year
  Contract_Two year
  PaperlessBilling_Yes
  PaymentMethod_Bank transfer (automatic)
  PaymentMethod_Credit card (automatic)
  PaymentMethod_Electronic check
  PaymentMethod_Mailed check


In [43]:
# Handle class imbalance

from imblearn.over_sampling import SMOTE

print(f"before SMOTE - Train set churn distribution:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_processed, y_train)

print(f"\nAfter SMOTE — Train set churn distribution:")
print(y_train_balanced.value_counts())
print(f"\nNew training set shape: {X_train_balanced.shape}")
print(f"New traingig target set shape: {y_train_balanced.shape}")

before SMOTE - Train set churn distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

After SMOTE — Train set churn distribution:
Churn
0    4139
1    4139
Name: count, dtype: int64

New training set shape: (8278, 26)
New traingig target set shape: (8278,)


Random oversampling duplicates existing rows — model memorizes them. SMOTE interpolates between existing minority samples to create realistic new ones. Better generalization.

We use SMOTE — Synthetic Minority Over-sampling Technique. It creates synthetic churner samples so the model trains on a balanced dataset.

Important: SMOTE applied only to training data. Test set stays untouched — it represents real-world distribution.

In [44]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed arrays
np.save('../data/processed/X_train.npy', X_train_balanced)
np.save('../data/processed/X_test.npy', X_test_processed)
np.save('../data/processed/y_train.npy', y_train_balanced)
np.save('../data/processed/y_test.npy', y_test.values)

# Save feature names for SHAP
pd.Series(all_feature_names).to_csv('../data/processed/feature_names.csv',header = ["features"], index=False)

# Save the fitted preprocessor — the API will load this
joblib.dump(preprocessor, '../models/preprocessor.pkl')

print("✅ Saved:")
print("  data/processed/X_train.npy")
print("  data/processed/X_test.npy")
print("  data/processed/y_train.npy")
print("  data/processed/y_test.npy")
print("  data/processed/feature_names.csv")
print("  models/preprocessor.pkl")

✅ Saved:
  data/processed/X_train.npy
  data/processed/X_test.npy
  data/processed/y_train.npy
  data/processed/y_test.npy
  data/processed/feature_names.csv
  models/preprocessor.pkl
